In [72]:
import requests                          # HTTP fetching
from bs4 import BeautifulSoup            # HTML parsing
import pandas as pd                      # Tabular data
import sqlite3                           # Local database
import time                              # Polite pauses between requests
import os
from datetime import datetime

print('✅ All libraries imported successfully')

✅ All libraries imported successfully


In [74]:
# ── Always check status before scraping ──────────────────────────────────
def safe_get(url, delay=1):
    """Fetch a URL, return None if it fails, and sleep politely after."""
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()    # Raises exception for 4xx / 5xx codes
        time.sleep(delay)              # Polite pause — never hammer a server
        return response
    except requests.RequestException as e:
        print(f'❌ Request failed: {e}')
        return None

resp = safe_get('https://books.toscrape.com')
if resp:
    print(f'✅ Got {len(resp.text):,} chars')

✅ Got 51,294 chars


In [75]:
with open('books.toscrape.html', 'r', encoding='utf-8') as f:
    html = f.read()

print(f'HTML file loaded: {len(html):,} characters')
soup = BeautifulSoup(html, 'html.parser')
print('BeautifulSoup object created ✅')

HTML file loaded: 51,274 characters
BeautifulSoup object created ✅


In [76]:
resp = safe_get('https://books.toscrape.com/catalogue/page-1.html')
soup = BeautifulSoup(resp.text, 'html.parser')


books_cards = soup.find_all('div', class_='page_inner')
print(f'Found {len(books_cards)} books cards')

# Preview the first card's raw HTML
print('\n--- First card structure ---')
print(books_cards[0].prettify()[:600])

Found 2 books cards

--- First card structure ---
<div class="page_inner">
 <div class="row">
  <div class="col-sm-8 h1">
   <a href="../index.html">
    Books to Scrape
   </a>
   <small>
    We love being scraped!
   </small>
  </div>
 </div>
</div>



In [77]:
books_cards = soup.find_all('article', class_='product_pod')

print("Number of books:", len(books_cards))

Number of books: 20


In [78]:
print(books_cards[0].prettify())

<article class="product_pod">
 <div class="image_container">
  <a href="a-light-in-the-attic_1000/index.html">
   <img alt="A Light in the Attic" class="thumbnail" src="../media/cache/2c/da/2cdad67c44b002e7ead0cc35693c0e8b.jpg"/>
  </a>
 </div>
 <p class="star-rating Three">
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
 </p>
 <h3>
  <a href="a-light-in-the-attic_1000/index.html" title="A Light in the Attic">
   A Light in the ...
  </a>
 </h3>
 <div class="product_price">
  <p class="price_color">
   Â£51.77
  </p>
  <p class="instock availability">
   <i class="icon-ok">
   </i>
   In stock
  </p>
  <form>
   <button class="btn btn-primary btn-block" data-loading-text="Adding..." type="submit">
    Add to basket
   </button>
  </form>
 </div>
</article>



In [58]:
print("number of bookcards:", len(books_cards))

number of bookcards: 20


In [106]:
# ── Step 3: Extract data from one card (prototype before looping) ─────────
card = books_cards[0]   # First book

title  = card.find('h3') .a['title']
price  = card.find('p',    class_='price_color').get_text(strip=True)
availability = card.find('p', class_='instock availability').get_text(strip=True)
rating   = book.find('p', class_='star-rating')['class'][1]
page_number = soup.find('li', class_='current').get_text(strip=True)
scraped_at   = scraped_at = datetime.now()

print(f'Title    : {title}')
print(f'Price  : {price}')
print(f'Rating : {rating}')
print(f'Availability   : {availability}')
print(f'Page Number     : {page_number}')
print(f'Scraped at   : {scraped_at}')

Title    : A Light in the Attic
Price  : Â£51.77
Rating : Three
Availability   : In stock
Page Number     : Page 10 of 50
Scraped at   : 2026-10-02 00:57:08.050055


In [108]:
# ── Step 4: Loop over ALL book cards ────────────────────────────────────────
data = []

for card in books_cards:
    title    = card.find('h3') .a['title']
    price  = card.find('p',    class_='price_color').get_text(strip=True)
    availability = card.find('p', class_='instock availability').get_text(strip=True)
    rating   = book.find('p', class_='star-rating')['class'][1]
    page_number = soup.find('li', class_='current').get_text(strip=True)
    scraped_at   = scraped_at = datetime.now()

    data.append({
        'title'   : title,
        'price' : price,
        'rating': rating,
        'availability'  : availability,
        'page_number': page_number,
        'scraped_at'  : scraped_at   
    })

df = pd.DataFrame(data)
print(f'Scraped {len(df)} book cards')
df.head()

Scraped 20 book cards


,title,price,rating,availability,page_number,scraped_at
0,A Light in the Attic,Â£51.77,Three,In stock,Page 10 of 50,2026-10-02 00:57:24.160070
1,Tipping the Velvet,Â£53.74,Three,In stock,Page 10 of 50,2026-10-02 00:57:24.161578
2,Soumission,Â£50.10,Three,In stock,Page 10 of 50,2026-10-02 00:57:24.164825
3,Sharp Objects,Â£47.82,Three,In stock,Page 10 of 50,2026-10-02 00:57:24.165897
4,Sapiens: A Brief History of Humankind,Â£54.23,Three,In stock,Page 10 of 50,2026-10-02 00:57:24.166903


In [114]:
# ── Full pagination loop — 10 pages ────────────────────────────────────────
BASE_URL  = 'https://books.toscrape.com/catalogue/page-{}.html'
rating_score = {'One' : 1,
                'Two' : 2,
                'Three' : 3,
                'Four' : 4,
                'Five' : 5


}


all_books = []

for page_num in range(1, 11):      # Pages 1, 2, 3
    print(f'  📖 Fetching page {page_num}...')

    resp = safe_get(BASE_URL.format(page_num), delay=1)
    if resp is None:
        print(f'  ⚠️  Skipping page {page_num} — request failed')
        continue

    soup = BeautifulSoup(resp.text, 'html.parser')

    for book in soup.find_all('article', class_='product_pod'):
        title    = book.find('h3') .a['title']
        price  = book.find('p',    class_='price_color').get_text(strip=True)
        availability = book.find('p', class_='instock availability').get_text(strip=True)
        rating   = book.find('p', class_='star-rating')['class'][1]
        page_number = soup.find('li', class_='current').get_text(strip=True)
        scraped_at   = scraped_at = datetime.now()

        all_books.append({
            'title'   : title,
            'price' : price,
            'rating': rating,
            'availability'  : availability,
            'page_number': page_number,
            'scraped_at'  : scraped_at  
        })

books_df = pd.DataFrame(all_books)
print(f'\n✅ Scraped {len(books_df)} books across 10 pages')
books_df.head(25)

  📖 Fetching page 1...
  📖 Fetching page 2...
  📖 Fetching page 3...
  📖 Fetching page 4...
  📖 Fetching page 5...
  📖 Fetching page 6...
  📖 Fetching page 7...
  📖 Fetching page 8...
  📖 Fetching page 9...
  📖 Fetching page 10...

✅ Scraped 200 books across 10 pages


,title,price,rating,availability,page_number,scraped_at
0,A Light in the Attic,Â£51.77,Three,In stock,Page 1 of 50,2026-10-02 01:06:41.242239
1,Tipping the Velvet,Â£53.74,One,In stock,Page 1 of 50,2026-10-02 01:06:41.242897
2,Soumission,Â£50.10,One,In stock,Page 1 of 50,2026-10-02 01:06:41.243551
3,Sharp Objects,Â£47.82,Four,In stock,Page 1 of 50,2026-10-02 01:06:41.244219
4,Sapiens: A Brief History of Humankind,Â£54.23,Five,In stock,Page 1 of 50,2026-10-02 01:06:41.244919
5,The Requiem Red,Â£22.65,One,In stock,Page 1 of 50,2026-10-02 01:06:41.245551
6,The Dirty Little Secrets of Getting Your Dream...,Â£33.34,Four,In stock,Page 1 of 50,2026-10-02 01:06:41.246124
7,The Coming Woman: A Novel Based on the Life of...,Â£17.93,Three,In stock,Page 1 of 50,2026-10-02 01:06:41.246810
8,The Boys in the Boat: Nine Americans and Their...,Â£22.60,Four,In stock,Page 1 of 50,2026-10-02 01:06:41.247749
9,The Black Maria,Â£52.15,One,In stock,Page 1 of 50,2026-10-02 01:06:41.248367


In [110]:
# ── Save and inspect ──────────────────────────────────────────────────────
books_df.to_csv('books_10pages.csv', index=False)
print('Saved to books_10pages.csv')

# Distribution by rating
print('\nBooks by rating:')
print(books_df['rating'].value_counts())

# Price column is a string — quick cleanup
books_df['price_num'] = books_df['price'].str.replace('Â£', '', regex=False).str.replace('£', '', regex=False).astype(float)
print(f'\nPrice range: £{books_df["price_num"].min()} – £{books_df["price_num"].max()}')
print(f'Average price: £{books_df["price_num"].mean():.2f}')

Saved to books_10pages.csv

Books by rating:
rating
One      49
Two      41
Four     38
Five     37
Three    35
Name: count, dtype: int64

Price range: £10.16 – £59.64
Average price: £34.80


In [92]:
books_df ['page_number'].value_counts()

page_number
Page 1 of 50     20
Page 2 of 50     20
Page 3 of 50     20
Page 4 of 50     20
Page 5 of 50     20
Page 6 of 50     20
Page 7 of 50     20
Page 8 of 50     20
Page 9 of 50     20
Page 10 of 50    20
Name: count, dtype: int64

In [116]:
books_df

,title,price,rating,availability,page_number,scraped_at
0,A Light in the Attic,Â£51.77,Three,In stock,Page 1 of 50,2026-10-02 01:06:41.242239
1,Tipping the Velvet,Â£53.74,One,In stock,Page 1 of 50,2026-10-02 01:06:41.242897
2,Soumission,Â£50.10,One,In stock,Page 1 of 50,2026-10-02 01:06:41.243551
3,Sharp Objects,Â£47.82,Four,In stock,Page 1 of 50,2026-10-02 01:06:41.244219
4,Sapiens: A Brief History of Humankind,Â£54.23,Five,In stock,Page 1 of 50,2026-10-02 01:06:41.244919
...,...,...,...,...,...,...
195,Eureka Trivia 6.0,Â£54.59,Four,In stock,Page 10 of 50,2026-10-02 01:06:56.350541
196,Drive: The Surprising Truth About What Motivat...,Â£34.95,Four,In stock,Page 10 of 50,2026-10-02 01:06:56.351083
197,Done Rubbed Out (Reightman & Bailey #1),Â£37.72,Five,In stock,Page 10 of 50,2026-10-02 01:06:56.351622
198,Doing It Over (Most Likely To #1),Â£35.61,Three,In stock,Page 10 of 50,2026-10-02 01:06:56.352179


In [123]:
print("Shape:", books_df.dtypes)

Shape: title                      str
price                      str
rating                     str
availability               str
page_number                str
scraped_at      datetime64[us]
dtype: object
